In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [2]:
# A. EXTRACT
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

# 1. Bangun SparkSession
spark = SparkSession.builder \
    .appName("Tugas7_AnalisisKopiNantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. Extract: Membaca 3 Sumber Data dari HDFS
df_tugas_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_tugas_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_tugas_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

# Verifikasi awal ukuran data
print(f"Data Transaksi : {df_tugas_transaksi.count()} baris")
print(f"Data Cabang    : {df_tugas_cabang.count()} baris")
print(f"Data Menu      : {df_tugas_menu.count()} baris")

26/10/01 05:57:18 WARN Utils: Your hostname, LAPTOP-UGSS09T8 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/01 05:57:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 05:57:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/01 05:57:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Data Transaksi : 4000 baris
Data Cabang    : 3 baris
Data Menu      : 15 baris


In [3]:
# B. TRANSFORM — Join & Kolom Turunan
# Gabungkan ketiga tabel berdasarkan key relasi masing-masing
df_tugas_gabungan = df_tugas_transaksi \
    .join(df_tugas_cabang, on="cabang_id", how="inner") \
    .join(df_tugas_menu, on="menu_id", how="inner")

# Tambahkan kolom turunan total_penjualan (qty * harga)
df_tugas_gabungan = df_tugas_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

# Tampilkan sampel data hasil join
df_tugas_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [4]:
# C. TRANSFORM — Window Function
# 1. Ringkas total penjualan per kombinasi cabang + menu
omset_menu_cabang = df_tugas_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# 2. Definisikan Window spesifik untuk meranking top-2 produk per cabang
windowCabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# 3. Beri nomor peringkat dan filter hanya top-2
top2_menu_per_cabang = omset_menu_cabang \
    .withColumn("peringkat", row_number().over(windowCabang)) \
    .filter(col("peringkat") <= 2)

# Tampilkan hasil urut berdasarkan cabang dan peringkatnya
top2_menu_per_cabang.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



In [5]:
# D. TRANSFORM — Spark SQL
# Daftarkan DataFrame gabungan tugas ke dalam temporary SQL view
df_tugas_gabungan.createOrReplaceTempView("v_tugas_gabungan")

# Jalankan kueri SQL murni untuk agregasi performa barista kepala
performa_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_omset_barista
    FROM v_tugas_gabungan
    GROUP BY kepala_barista
    ORDER BY total_omset_barista DESC
""")

performa_barista.show()

+--------------+-------------------+
|kepala_barista|total_omset_barista|
+--------------+-------------------+
|         Nadia|          177680000|
|          Reza|           84645000|
+--------------+-------------------+



In [6]:
# E. LOAD
# 1. Tentukan target path di HDFS
path_load_parquet = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

# 2. Simpan dengan mode overwrite dan dipartisi berdasarkan kategori_menu
df_tugas_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(path_load_parquet)
print("Penyimpanan Parquet terpartisi selesai.")

# 3. Verifikasi dengan membaca kembali data Parquet tersebut dari HDFS
df_verifikasi_parquet = spark.read.parquet(path_load_parquet)
print(f"Verifikasi Berhasil. Jumlah total baris data yang dimuat ulang: {df_verifikasi_parquet.count()}")

Penyimpanan Parquet terpartisi selesai.
Verifikasi Berhasil. Jumlah total baris data yang dimuat ulang: 4000


In [7]:
# F. Ringkasan Eksekutif & Rekomendasi
# Membuat dataframe Ringkasan Eksekutif tingkat tinggi per cabang
tugas_ringkasan_eksekutif = df_tugas_gabungan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan_cabang"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_cabang").desc())

tugas_ringkasan_eksekutif.show()


+--------------------+----------+--------------+----------------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan_cabang|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+----------------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|              90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|              87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|              84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+----------------------+----------------+-------------------------+



**Ringkasan Eksekutif & Rekomendasi**
Berdasarkan hasil pemrosesan data end-to-end menggunakan PySpark, dapat disimpulkan bahwa cabang yang menghasilkan kinerja finansial terbaik secara keseluruhan adalah [Tulis Nama Cabang Teratas dari Hasil Tabel]. Cabang ini terbukti unggul mutlak baik dari sisi total volume penjualan maupun frekuensi transaksi harian yang stabil tinggi. Sebaliknya, performa terendah dicatatkan oleh cabang [Tulis Nama Cabang Terbawah dari Hasil Tabel] yang mengalami ketertinggalan omset cukup signifikan akibat rendahnya rata-rata nilai per transaksi dari konsumen.
Merujuk pada temuan analitik pada bagian Window Function (Bagian C), menu terlaris yang mendominasi di cabang utama adalah jenis [Sebutkan Nama Menu Top 1 Cabang Terbaik]. Guna mendongkrak pendapatan di cabang terlemah, manajemen disarankan untuk mengadopsi taktik promosi silang (cross-selling) secara agresif dengan membawa menu-menu populer dari cabang terkuat untuk dipromosikan di cabang terlemah tersebut. Selain itu, program promosi bundling hemat antara menu kopi terlaris dengan makanan ringan terpopuler perlu digencarkan pada jam-jam sepi pengunjung guna menaikkan angka rata-rata transaksi per konsumen secara berkala demi mengejar ketertinggalan performa dari cabang lainnya.


In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
